In [1]:
# import modules
import numpy as np
import random
import math
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm
from time import time
#import arviz as az
import tensorflow as tf
import tensorflow_probability as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

np.set_printoptions(suppress=True)

from mssf import stepSelectionVI


## Load data

In [2]:
 
nbObs=1000001 #nbObs Number of observations
beta=np.array([[-1.5,1.8]]) # Vector of resource selection coefficients
L = 50.0

pos_filename  = "pos_N_" + str(nbObs-1) + "_b0_" + str(beta[0][0]) + "_b1_" + str(beta[0][1])+ ".csv"
grid_filename  = "cov_N_" + str(nbObs-1) + "_b0_" + str(beta[0][0]) + "_b1_" + str(beta[0][1])+ ".npy"


df = pd.read_csv("../synthetic_data/" + pos_filename)

xy1 = df[["x","y"]].values #,["y"]].values
dt = df["time"].values #,["y"]].values
ID = df["ID"].values #,["y"]].values
#df

#print(xy1.shape)
#print(cov_cube.shape)
cov_cube = np.load("../synthetic_data/" + grid_filename)
print(cov_cube.shape)


(2, 51, 51)


## Process to convert to steps

In [3]:
# preprocessing steps convert to tensors and handle the change of ID, start/end points and time between fixes
start_points = []
end_points = []
step_times = []
for i in np.unique(ID):
    cxy = xy1[ID==i]#[:1024*100+1]
    cdt = dt[ID==i]#[:1024*100+1]
    if cdt.shape[0]<2:
        continue
    start_points.append(cxy[:-1])
    end_points.append(cxy[1:])
    step_times.append(np.atleast_2d(cdt[:-1]).T)

    
indexes = np.arange(np.vstack(start_points).shape[0])
np.random.shuffle(indexes)
start_points = tf.convert_to_tensor(np.vstack(start_points)[indexes],dtype=tf.float32)
end_points = tf.convert_to_tensor(np.vstack(end_points)[indexes],dtype=tf.float32)
step_times = tf.convert_to_tensor(np.vstack(step_times)[indexes],dtype=tf.float32)


cov_tensor = tf.transpose(tf.convert_to_tensor(cov_cube,dtype=tf.float32),[2,1,0])

x_ref_min = [0., 0.]
x_ref_max = [L, L]

2022-08-17 17:23:00.138428: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-08-17 17:23:07.383928: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1510] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15391 MB memory:  -> device: 0, name: Quadro GP100, pci bus id: 0000:3b:00.0, compute capability: 6.0


## Create the instance of the model and set up the training

In [4]:
ssf = stepSelectionVI(2,cov_tensor,move_std=0.96,L=50.0,n_gh_points=4,n_vi_samples=64)

In [5]:

## set up the dataset and optimizer
batch_size=1024
train_dataset = tf.data.Dataset.from_tensor_slices((start_points, end_points, step_times))
train_dataset = train_dataset.shuffle(buffer_size=64).batch(batch_size,drop_remainder=True)
#train_dataset = train_dataset.batch(batch_size,drop_remainder=True)

kl_weight = batch_size/start_points.shape[0]
#optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001)

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(0.1,decay_steps=100, decay_rate=0.9, staircase=True)

optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule)


In [6]:
def plot_fit():# data_batch
    print('move std ' + str(ssf.move_std.numpy()) + 'betas ' + str(ssf.beta_mean.numpy()) + ' stds: ', str(ssf.beta_std.numpy().flatten()))

    b1mean = ssf.beta_mean.numpy()[0]
    b1scale = ssf.beta_std.numpy()[0,0,0]
    b2mean = ssf.beta_mean.numpy()[1]
    b2scale = ssf.beta_std.numpy()[0,1,1]

    fig,ax=plt.subplots(1,2, figsize=(14,5))


    x = np.arange(b1mean-b1scale*5,b1mean+b1scale*5,0.001)
    y = tfp.distributions.Normal(loc=b1mean,scale=b1scale).prob(x)

    ax[0].plot(x,y,c='C0')
    ax[0].fill_between(x,y,color='C0',alpha=0.5)
    ax[0].axvline(-1.5,c='C1')
    ax[0].set_ylim(0,y.numpy().max()*1.2)


    x = np.arange(b2mean-b2scale*5,b2mean+b2scale*5,0.001)
    y = tfp.distributions.Normal(loc=b2mean,scale=b2scale).prob(x)

    ax[1].plot(x,y,c='C0')
    ax[1].fill_between(x,y,color='C0',alpha=0.5)

    ax[1].axvline(1.8,c='C1')
    ax[1].set_ylim(0,y.numpy().max()*1.2)
    #plt.savefig('1millionpoints.png')
    plt.show()


## Run the training


In [ ]:
epochs=range(5)

for epoch in epochs:
    epoch_loss = 0.0
    for data_batch in tqdm(train_dataset):
        with tf.GradientTape() as tape:
            loss = ssf.variational_loss(*data_batch, kl_weight)
        epoch_loss+=np.squeeze(loss.numpy())
        gradients=tape.gradient(loss,ssf.trainable_variables)
        optimizer.apply_gradients(zip(gradients, ssf.trainable_variables))
        #break
    plot_fit()
    #if epoch % 1 == 0: 
        #print('Epoch ' + str(epoch) + ' complete. Loss: ', epoch_loss)
        #print('Epoch ' + str(epoch) +' betas ' + str(ssf2.beta_mean.numpy()) + ' stds: ', str(ssf2.beta_std.numpy().flatten()))
    

  0%|          | 0/976 [00:00<?, ?it/s]2022-08-17 17:23:17.209786: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)
2022-08-17 17:23:51.913334: W tensorflow/python/util/util.cc:348] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.
  1%|          | 11/976 [02:33<2:11:39,  8.19s/it]